# Clean Length Measurement Analysis

**Data channels:**
- Channel 1: Length measurement (voltage)
- Channel 2: Force signal (voltage)
- Control voltage: Expected square wave (-0.8V → +0.6V, 300s each)

**Linear relationship:** `length (mm) = -0.04130 × voltage`

**Analysis:** 3 focused figures with drift correction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from sklearn.linear_model import LinearRegression
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = [14, 8]

In [ ]:
# Load and prepare data
df = pd.read_csv('20260430_fixedforce_length_measurement.csv')
print(f"Loaded {len(df)} data points over {df['elapsed_s'].max()/60:.1f} minutes")

# Apply median filtering to reduce noise
filter_size = 15
df['length_filtered'] = median_filter(df['ch1_MEAN_V'], size=filter_size)
df['force_filtered'] = median_filter(df['ch2_MEAN_V'], size=filter_size)

# Conversion factor: length (mm) = -0.04130 * voltage
length_conversion = -0.04130
df['length_mm'] = length_conversion * df['length_filtered']

print(f"Length range: {df['length_mm'].min():.2f} to {df['length_mm'].max():.2f} mm")
print(f"Voltage range: {df['ch1_MEAN_V'].min():.3f} to {df['ch1_MEAN_V'].max():.3f} V")

In [ ]:
# Detect cycle start (when square wave begins)
# Look for significant changes in the expected pattern
force_diff = np.abs(np.diff(df['force_filtered']))
change_threshold = np.percentile(force_diff, 95)  # Top 5% of changes
significant_changes = np.where(force_diff > change_threshold)[0]

if len(significant_changes) > 0:
    cycle_start_idx = significant_changes[0]
    cycle_start_time = df['elapsed_s'].iloc[cycle_start_idx]
    print(f"Detected cycle start at: {cycle_start_time:.1f} seconds")
else:
    cycle_start_time = 30  # Fallback estimate
    print(f"Using estimated cycle start: {cycle_start_time:.1f} seconds")

# Create control voltage pattern (expected square wave)
control_voltage = np.zeros_like(df['elapsed_s'])
cycle_mask = df['elapsed_s'] >= cycle_start_time

for i, t in enumerate(df['elapsed_s']):
    if t >= cycle_start_time:
        time_since_start = t - cycle_start_time
        cycle_position = time_since_start % 600  # 600s = one complete cycle
        
        if cycle_position < 300:
            control_voltage[i] = -0.8  # First 300s
        else:
            control_voltage[i] = 0.6   # Next 300s
    else:
        control_voltage[i] = 0.0  # Before cycles start

df['control_voltage'] = control_voltage
print(f"Created control voltage pattern: {np.unique(control_voltage)}V")

In [ ]:
# Drift correction: sample every 600 seconds and fit linear trend
cycle_data = df[df['elapsed_s'] >= cycle_start_time].copy()
cycle_times = cycle_data['elapsed_s'] - cycle_start_time

# Sample points every 600 seconds (complete cycles)
sampling_interval = 600
max_cycle_time = cycle_times.iloc[-1]
sample_times = np.arange(0, max_cycle_time + sampling_interval, sampling_interval)

sample_values = []
sample_timestamps = []

for target_time in sample_times:
    if target_time <= max_cycle_time:
        # Find closest data point
        time_diffs = np.abs(cycle_times - target_time)
        closest_idx = time_diffs.idxmin()
        closest_time = cycle_times.loc[closest_idx]
        closest_value = cycle_data.loc[closest_idx, 'length_filtered']
        
        sample_values.append(closest_value)
        sample_timestamps.append(closest_time)

# Fit linear drift trend
if len(sample_values) >= 2:
    sample_values = np.array(sample_values)
    sample_timestamps = np.array(sample_timestamps)
    
    drift_model = LinearRegression()
    drift_model.fit(sample_timestamps.reshape(-1, 1), sample_values)
    
    drift_slope = drift_model.coef_[0]
    drift_intercept = drift_model.intercept_
    
    # Apply drift correction
    drift_correction = drift_slope * cycle_times + drift_intercept
    cycle_data['length_drift_corrected'] = cycle_data['length_filtered'] - drift_correction + drift_intercept
    cycle_data['length_mm_corrected'] = length_conversion * cycle_data['length_drift_corrected']
    
    # Add to main dataframe
    df['length_drift_corrected'] = df['length_filtered'].copy()
    df.loc[cycle_data.index, 'length_drift_corrected'] = cycle_data['length_drift_corrected']
    df['length_mm_corrected'] = length_conversion * df['length_drift_corrected']
    
    print(f"Drift correction applied:")
    print(f"  Drift rate: {drift_slope:.6f} V/s ({drift_slope*3600:.4f} V/hour)")
    print(f"  Total drift: {drift_slope * max_cycle_time:.6f}V over {max_cycle_time/60:.1f} minutes")
else:
    print("Not enough sample points for drift correction")
    df['length_drift_corrected'] = df['length_filtered']
    df['length_mm_corrected'] = df['length_mm']

In [ ]:
# FIGURE 1: Length voltage over time with dual axes and control voltage overlay
fig, ax1 = plt.subplots(figsize=(16, 8))

# Primary plot: Length voltage (filtered)
color = 'tab:blue'
ax1.set_xlabel('Time (seconds)', fontsize=14)
ax1.set_ylabel('Length Voltage (V)', color=color, fontsize=14)
line1 = ax1.plot(df['elapsed_s'], df['length_filtered'], color=color, linewidth=2, 
                 alpha=0.8, label='Length (Filtered)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Secondary Y-axis: Length in mm
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Length (mm)', color=color, fontsize=14)
ax2.plot(df['elapsed_s'], df['length_mm'], color=color, linewidth=2, alpha=0.8, label='Length (mm)')
ax2.tick_params(axis='y', labelcolor=color)

# Overlay control voltage on top
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 60))
color = 'tab:red'
ax3.set_ylabel('Control Voltage (V)', color=color, fontsize=14)
ax3.plot(df['elapsed_s'], df['control_voltage'], color=color, linewidth=3, 
         alpha=0.7, label='Control Square Wave', linestyle='--')
ax3.tick_params(axis='y', labelcolor=color)
ax3.set_ylim(-1.2, 1.0)

# Mark cycle start
ax1.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)
ax1.text(cycle_start_time + 10, ax1.get_ylim()[1]*0.9, f'Cycle Start\n({cycle_start_time:.0f}s)', 
         fontsize=12, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax1.set_title('Length Measurement with Control Voltage Pattern', fontsize=16, fontweight='bold', pad=20)

# Create combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax3.get_legend_handles_labels()
ax1.legend(lines1 + lines2 + lines3, labels1 + labels2 + labels3, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 2: Drift corrected fit (red line) - just the correction
fig, ax = plt.subplots(figsize=(16, 8))

if len(sample_values) >= 2:
    # Plot original data (faded)
    cycle_times_plot = cycle_data['elapsed_s']
    ax.plot(cycle_times_plot, cycle_data['length_filtered'], alpha=0.3, 
            color='lightblue', linewidth=1, label='Original (Filtered)')
    
    # Plot drift line
    drift_line_full = drift_slope * cycle_times + drift_intercept
    ax.plot(cycle_times_plot, drift_line_full, color='red', linewidth=4, 
            label=f'Linear Drift Trend\n(slope = {drift_slope:.6f} V/s)', alpha=0.9)
    
    # Mark sample points
    sample_cycle_times = [cycle_data.iloc[i]['elapsed_s'] for i, t in enumerate(cycle_times) 
                         if t in sample_timestamps]
    ax.scatter(sample_cycle_times[:len(sample_values)], sample_values, 
               color='red', s=100, zorder=5, edgecolors='darkred', linewidth=2,
               label=f'Sample Points (every 600s)')
    
    ax.set_xlabel('Time (seconds)', fontsize=14)
    ax.set_ylabel('Length Voltage (V)', fontsize=14)
    ax.set_title('Drift Correction: Linear Trend Fit', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Add drift statistics
    textstr = f'Drift Rate: {drift_slope*3600:.4f} V/hour\nTotal Drift: {drift_slope * max_cycle_time:.6f}V\nOver {max_cycle_time/60:.1f} minutes'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props)
else:
    ax.text(0.5, 0.5, 'Not enough data for drift analysis', 
            transform=ax.transAxes, fontsize=16, ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 3: Drift-corrected data with dual axes and control voltage subfigure
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.3)

# Main plot: Drift-corrected length data
ax1 = fig.add_subplot(gs[0, 0])

# Original data (faded)
color = 'lightblue'
ax1.plot(df['elapsed_s'], df['length_filtered'], color=color, linewidth=1, 
         alpha=0.4, label='Original (Filtered)')

# Drift-corrected data (prominent)
color = 'tab:blue'
ax1.set_ylabel('Length Voltage (V)', color=color, fontsize=14)
ax1.plot(df['elapsed_s'], df['length_drift_corrected'], color=color, linewidth=2.5, 
         alpha=0.9, label='Drift Corrected')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Secondary Y-axis: Length in mm (corrected)
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Length (mm) - Corrected', color=color, fontsize=14)
ax2.plot(df['elapsed_s'], df['length_mm_corrected'], color=color, linewidth=2.5, 
         alpha=0.9, label='Length (mm) - Corrected')
ax2.tick_params(axis='y', labelcolor=color)

# Mark cycle start
ax1.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)

ax1.set_title('Drift-Corrected Length Measurements', fontsize=16, fontweight='bold')

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Subfigure: Control voltage pattern
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(df['elapsed_s'], df['control_voltage'], color='red', linewidth=3, 
         alpha=0.8, label='Control Square Wave')
ax3.axvline(x=cycle_start_time, color='black', linestyle=':', alpha=0.8, linewidth=2)
ax3.set_xlabel('Time (seconds)', fontsize=14)
ax3.set_ylabel('Control (V)', fontsize=12)
ax3.set_title('Control Voltage Pattern (-0.8V → +0.6V, 300s each)', fontsize=14)
ax3.grid(True, alpha=0.3)
ax3.set_ylim(-1.2, 1.0)

# Add cycle annotations
if cycle_start_time > 0:
    cycle_centers = np.arange(cycle_start_time + 150, df['elapsed_s'].max(), 300)  # Every 300s
    for i, center in enumerate(cycle_centers[:8]):  # First 8 cycles
        if i % 2 == 0:
            ax3.annotate('-0.8V', xy=(center, -0.8), xytext=(center, -1.1),
                        ha='center', fontsize=10, alpha=0.7)
        else:
            ax3.annotate('+0.6V', xy=(center, 0.6), xytext=(center, 0.8),
                        ha='center', fontsize=10, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("ANALYSIS SUMMARY")
print("=" * 50)
print(f"📊 Total duration: {df['elapsed_s'].max()/60:.1f} minutes")
print(f"⏱️  Cycle start: {cycle_start_time:.1f} seconds")
print(f"🔧 Median filter size: {filter_size} points")

if len(sample_values) >= 2:
    print(f"\n📈 DRIFT CORRECTION:")
    print(f"   Rate: {drift_slope*3600:.4f} V/hour")
    print(f"   Total: {drift_slope * max_cycle_time:.6f}V")
    
    # Noise reduction
    original_std = cycle_data['length_filtered'].std()
    corrected_std = cycle_data['length_drift_corrected'].std()
    reduction = (original_std - corrected_std) / original_std * 100
    print(f"   Std reduction: {reduction:.1f}%")

print(f"\n📏 LENGTH CONVERSION:")
print(f"   Formula: length (mm) = {length_conversion:.5f} × voltage (V)")
print(f"   Range: {df['length_mm_corrected'].min():.2f} to {df['length_mm_corrected'].max():.2f} mm")
print(f"   Total span: {df['length_mm_corrected'].max() - df['length_mm_corrected'].min():.2f} mm")

print(f"\n🎛️  CONTROL PATTERN:")
print(f"   Voltage levels: -0.8V and +0.6V")
print(f"   Cycle period: 600 seconds (300s each level)")
control_cycles = (df['elapsed_s'].max() - cycle_start_time) / 600
print(f"   Complete cycles: {control_cycles:.1f}")